# Step 5 - Run in Google Colab

Upload the complete project ZIP when prompted. This notebook uses the saved Step 4 results and therefore does not download or rerun the sentence-transformer model.

In [ ]:
# 1. Upload and extract the complete project ZIP
from google.colab import files
from pathlib import Path
import shutil, sys

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the complete project ZIP file.')

extract_dir = Path('/content/study_to_career')
extract_dir.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive('/content/' + zip_name, extract_dir)
matches = list(extract_dir.rglob('step5_matching/matching.py'))
if not matches:
    raise FileNotFoundError('step5_matching/matching.py was not found in the ZIP.')
STEP5 = matches[0].parent
ROOT = STEP5.parent
sys.path.insert(0, str(STEP5))
print('Project root:', ROOT)

In [ ]:
# 2. Install the small Step 5 dependencies
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(STEP5 / 'requirements.txt')])

In [ ]:
# 3. Build profiles and run the 20-unit demonstration
import pandas as pd
from matching import (build_career_profiles, study_coverage_from_hits,
                      match_careers, learning_recommendations, sensitivity_analysis, app_payload)

jobs = pd.read_csv(ROOT / 'step1_data/job_cleaned.csv')
job_hits = pd.read_csv(ROOT / 'step4_semantic/results/hybrid_job_skills/job_skills_hybrid_long.csv')
study_hits = pd.read_csv(ROOT / 'step4_semantic/results/details/curriculum_all_hits.csv')
career_profiles = build_career_profiles(job_hits, jobs)
study_profile = study_coverage_from_hits(study_hits)
ranking, skill_details = match_careers(study_profile, career_profiles, top_n=30)
ranking[['rank', 'career', 'match_percent', 'n_covered_top_skills']]

In [ ]:
# 4. Inspect explanations for the highest-ranked career
best = ranking.iloc[0]
print('Recommended career:', best.career)
print('Matched skills:', ', '.join(best.matched_skills))
print('Priority gaps:', ', '.join(best.priority_gaps))
display(skill_details.query('career == @best.career').sort_values('contribution', ascending=False).head(10))
print('Recommended learning plan:')
learning_recommendations(skill_details, best.career, k=5)

In [ ]:
# 5. Check whether the ranking changes under top-20, top-30 and top-40
sensitivity_analysis(study_profile, career_profiles)

In [ ]:
# 6. Save and download the outputs
import json
out = Path('/content/step5_outputs')
out.mkdir(exist_ok=True)
career_profiles.to_csv(out / 'career_profiles.csv', index=False)
ranking.to_csv(out / 'career_ranking.csv', index=False)
skill_details.to_csv(out / 'skill_explanations.csv', index=False)
plans = pd.concat([learning_recommendations(skill_details, c) for c in ranking.career], ignore_index=True)
plans.to_csv(out / 'learning_recommendations.csv', index=False)
(out / 'app_payload.json').write_text(json.dumps(app_payload(ranking, skill_details), indent=2), encoding='utf-8')
archive = shutil.make_archive('/content/step5_outputs', 'zip', out)
files.download(archive)

## Run only selected courses

Replace the Step 3 line `study_profile = study_coverage_from_hits(study_hits)` with:

```python
selected = ['UTS 36118 Applied NLP', 'UTS 36120 Adv ML Application']
study_profile = study_coverage_from_hits(study_hits, unit_ids=selected)
```

For a new user-uploaded PDF, Member 4's live semantic extractor must first convert the PDF into a StudyProfile. Then call `study_coverage_from_profile(profile)` from `matching.py`.